![Ironhack logo](https://user-images.githubusercontent.com/23629340/40541063-a07a0a8a-601a-11e8-91b5-2f13e4e6b441.png)

# 02 | Pipelines and Models

*Project 2 — Machine Learning: From Data to Predictions*

**Team: Zoé Doucet**
**Dataset: Covertype**

---

Steps 5 to 8 of the workflow: build the pipeline, compare models fairly, tune the best, and choose the threshold. Everything here uses the training set only.

> **Done when:** you have a comparison table of at least three model families on the same folds against the baseline, the best one or two tuned, and a final model chosen for reasons you can state.

The sections below are a suggested route, not a form to fill in. Add sections, drop them, reorder them — what is graded is the work, not whether you kept the scaffold.

---
## 0. Setup

In [ ]:
import sys
sys.path.append("..")          # so `src` is importable from inside notebooks/

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (f1_score, recall_score, precision_score, accuracy_score,
                             classification_report, ConfusionMatrixDisplay, make_scorer)
from sklearn.dummy import DummyClassifier
from sklearn.model_selection import cross_validate, StratifiedKFold, train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline, make_pipeline
import io
from contextlib import redirect_stdout

from src.functions import *

pd.set_option("display.max_columns", 100)
sns.set_theme(style="whitegrid")

---
## 1. Load and split

The same loading and the same split as notebook 01. Put the code in `src/functions.py` so both notebooks use exactly the same rows.

In [ ]:
covertype= pd.read_csv("../data/covertype_clean.csv") # loading the data

In [19]:
# splitting the data using the split function
X, y, X_train, X_test, y_train, y_test, cv, folds = split( covertype.drop(columns= ["cover_type"]),
                                                    covertype["cover_type"],
                                                    0.25,
                                                    42,
                                                    5)

435759 training rows, 145253 test rows


---
## 2. The preprocessing pipeline

A `ColumnTransformer` with one branch per column type: impute, scale, encode. **Everything that learns from data goes inside**, so cross-validation refits it on every fold.

In [22]:
cat_features = ["soil_type", "wilderness_area"] # creating a lsit with the categorical features
num_features = X.drop(columns = cat_features).columns.tolist() # creating a list with the numerical features

preprocess = ColumnTransformer([
    ("num", 
      StandardScaler(), 
      num_features), # Scaling the numerical features
    ("cat", 
     OneHotEncoder(handle_unknown="ignore", sparse_output=False), 
     cat_features)])# transforming categrorical columns into binary columns


---
## 3. First model in the pipeline

One simple model at the end of the pipeline, scored with cross-validation. Is it better than the baseline, and by more than the spread across folds?

In [50]:
log_reg_1 = Pipeline([("prep", preprocess), ("model", LogisticRegression(max_iter=1000))])

In [51]:
score_log_reg, spread_log_reg = cross_validation (log_reg_1, X_train, y_train, folds)

Scores: {'f1': np.float64(0.533), 'recall': np.float64(0.512), 'accuracy': np.float64(0.725), 'precision': np.float64(0.604)} 
 Spread: {'f1': np.float64(0.005), 'recall': np.float64(0.005), 'accuracy': np.float64(0.002), 'precision': np.float64(0.008)}


The logistic regressionachieved a mean accuracy of 72.5 % which is higher than the baseline model. The model has an F1-score of 0.533, recall of 0.512, and precision of 0.604. The low standard deviations across folds (0.002–0.008) indicate consistent performance. However, the relatively low F1-score and recall suggest that the model struggles to identify all classes effectively.

---
## 4. Features

New features added inside the pipeline. Keep a feature only if it improves the cross-validated score, and write down what each one changed.

In [34]:
from sklearn.preprocessing import PolynomialFeatures, RobustScaler
from sklearn.feature_selection import SelectKBest, f_classif

To tune the model three different features will be testetd to see if they improve the model:
- RobustScaler: helps when numerical features contain outliers by scaling using the median and the interquantile range instead of the mean and standard deviation
- PolynomialFeatures: creates interactions between the numerical features and squares them
- SelectKBest: reduces noise and training time by selecting the k best features according to an ANOVA F-test

In [38]:
# RobustScaler
preprocess_robust = ColumnTransformer([
    ("num", 
     RobustScaler(), num_features), # changing the scaler from StandardScaler to RobustScaler
    ("cat", 
     OneHotEncoder(handle_unknown="ignore",sparse_output=True), cat_features)])

log_reg_rbsc = Pipeline([("prep", preprocess_robust),("model", LogisticRegression(max_iter=1000))])

scores_robust, spread_robust = cross_validation(log_reg_rbsc, X_train, y_train, folds)

Scores: {'f1': np.float64(0.532), 'recall': np.float64(0.511), 'accuracy': np.float64(0.725), 'precision': np.float64(0.603)} 
 Spread: {'f1': np.float64(0.005), 'recall': np.float64(0.005), 'accuracy': np.float64(0.002), 'precision': np.float64(0.009)}


In [ ]:
# This feature can not be used, as the modeling time is to long 
log_reg_pol = Pipeline([("prep", preprocess), ("poly", PolynomialFeatures(degree=2, include_bias=False)), ("model", LogisticRegression(solver="lbfgs", max_iter=200, C=1.0))])
scores_pol, spread_pol = cross_validation(log_reg_pol, X_train, y_train, folds)

In [46]:
best_score = 0
best_k = None
best_model = None

for k in [13, 20, 30, 40, "all"]:
    log_reg_slct = Pipeline([("prep", preprocess),("select", SelectKBest(score_func=f_classif, k=k)),("model", LogisticRegression(max_iter=1000))])

    with redirect_stdout(io.StringIO()):
        scores_slct, spread_slct = cross_validation(log_reg_slct, X_train, y_train, cv)

    if scores_slct["accuracy"] > best_score:
        best_score = scores_slct["accuracy"]
        best_k = k
        best_model = log_reg_slct
print(best_k)

all


In [53]:
import sys
sys.path.append("..")

import importlib
import src.functions

# Reload the module after you modify functions.py
importlib.reload(src.functions)

# Import the updated functions into your notebook
from src.functions import *

In [ ]:
models = {
    "Logistic Regression": (score_log_reg, spread_log_reg),
    "RobustScaler": (scores_robust,spread_robust),
    #"PolynomialFeatures": (score_pol, spread_pol), # was taken out because the model takes to long to run
    "SelectKBest": (scores_slct, spread_slct)}

compare_models (models)

,Model,f1,f1_spread,recall,recall_spread,accuracy,accuracy_spread,precision,precision_spread
0,Logistic Regression,0.533,0.005,0.512,0.005,0.725,0.002,0.604,0.008
1,RobustScaler,0.532,0.005,0.511,0.005,0.725,0.002,0.603,0.009
2,SelectKBest,0.533,0.005,0.512,0.005,0.725,0.002,0.604,0.008


SelectKBest and Logistic Regression are essentially the same because bets k is all columns.

---
## 5. Compare models

At least three model families on **the same folds**, with the mean and the spread. A table, and a chart of it.

In [ ]:
candidates = {
    "Logistic regression": make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)),
    "Random forest": RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1),
    "HistGradientBoosting": HistGradientBoostingClassifier(random_state=42),}

model_candidates = {}
for name, model in candidates.items():
    score, spread = cross_validation(model, X_train, y_train, folds)
    model_candidates[name] = (score, spread)

compare_models(model_candidates)

---
## 6. Tune

A `GridSearchCV` or `RandomizedSearchCV` on the best one or two, on the training set. Report the best parameters and the cross-validated score.

---
## 7. Threshold

Classification only. Cross-validated probabilities on the training set with `cross_val_predict`, then the threshold that fits your metric and the cost of each mistake. Regression: delete this section and say why in the README.

---
## 8. The final model

Which model, which parameters, which threshold, and why. If a simpler model is within the noise of the best one, say which you chose and why.